# 15 Data Mining Insights: Mysuru District PDS Ration Allotment Analysis

This notebook performs extensive data mining and statistical analysis on the **official 12-month Public Distribution System (PDS) ration allotment dataset** for all 9 taluks in Mysuru district (June 2025 – May 2026).

We extract **15 distinct facts, hidden trends, and statistical insights** from the database to evaluate administrative efficiency, demographic patterns, and distribution footprints.

---
## Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load dataset
df = pd.read_csv("mysuru_ration_dataset.csv")
print(f"Dataset Loaded. Shape: {df.shape}")
df.head()

## 1. Taluk-wise Ration Distribution Efficiency (Lifting Percentage)
We rank the taluks by their average **Lifting Percentage** to see which taluks are most and least efficient at distributing grains to cardholders.

In [ ]:
efficiency = df.groupby("Taluk_Name")["Lifting_Percentage"].mean().reset_index().sort_values(by="Lifting_Percentage", ascending=False)
print("Average Lifting Percentage:")
print(efficiency.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=efficiency, x="Lifting_Percentage", y="Taluk_Name", palette="viridis")
plt.title("Ration Distribution Efficiency by Taluk")
plt.xlabel("Average Lifting Percentage (%)")
plt.show()

## 2. Fair Price Shop (FPS) Workload and Card Density
We calculate the average **number of ration cards served per active Fair Price Shop (FPS)** to find which taluks have the highest workload per shop.

In [ ]:
fps_data = df[df["No_of_Active_FPS"] > 0].copy()
fps_data["Cards_Per_Shop"] = fps_data["No_of_Ration_Cards"] / fps_data["No_of_Active_FPS"]
workload = fps_data.groupby("Taluk_Name")["Cards_Per_Shop"].mean().reset_index().sort_values(by="Cards_Per_Shop", ascending=False)

print("Average Ration Cards served per Shop:")
print(workload.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=workload, x="Cards_Per_Shop", y="Taluk_Name", palette="coolwarm")
plt.title("Average Ration Cards Served per Fair Price Shop")
plt.xlabel("Ration Cards per Shop")
plt.show()

## 3. Active Card Utilization Rate
What percentage of registered ration cards actually collect their ration monthly?
$$\text{Utilization Rate} = \frac{\text{No. of Ration Cards Taken}}{\text{No. of Ration Cards}} \times 100$$

In [ ]:
util_data = df[df["No_of_Ration_Cards"] > 0].copy()
util_data["Card_Utilization_Rate"] = (util_data["No_of_Ration_Cards_Taken"] / util_data["No_of_Ration_Cards"]) * 100
utilization = util_data.groupby("Taluk_Name")["Card_Utilization_Rate"].mean().reset_index().sort_values(by="Card_Utilization_Rate", ascending=False)

print("Average Card Utilization Rate:")
print(utilization.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=utilization, x="Card_Utilization_Rate", y="Taluk_Name", palette="magma")
plt.title("Active Ration Card Utilization Rate (%)")
plt.show()

## 4. Aadhaar eKYC Verification rate among Active Beneficiaries
We measure the biometric verification compliance of active members:
$$\text{eKYC Rate} = \frac{\text{No. of Members Taken eKYC}}{\text{No. of Members Taken}} \times 100$$

In [ ]:
ekyc_data = df[df["No_of_Members_Taken"] > 0].copy()
ekyc_data["eKYC_Rate"] = (ekyc_data["No_of_Members_Taken_eKYC"] / ekyc_data["No_of_Members_Taken"]) * 100
ekyc_summary = ekyc_data.groupby("Taluk_Name")["eKYC_Rate"].mean().reset_index().sort_values(by="eKYC_Rate", ascending=False)

print("Average eKYC Verification Rate:")
print(ekyc_summary.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=ekyc_summary, x="eKYC_Rate", y="Taluk_Name", palette="plasma")
plt.title("Aadhaar eKYC Verification Rate (%)")
plt.xlim(95, 100)
plt.show()

## 5. Monthly Staple Foodgrain Allotment Trends
We plot the monthly allotment timeline for the main distributed staples (Rice and Ragi) across the district.

In [ ]:
month_order = ["June", "July", "August", "September", "October", "November", "December", "January", "February", "March", "April", "May"]
df['Month'] = pd.Categorical(df['Month'], categories=month_order, ordered=True)
monthly_allot = df.groupby(["Month", "Commodity_Name"])["Quantity_Allotted_NFSA_Qtls"].sum().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_allot[monthly_allot["Commodity_Name"].isin(["Rice", "Ragi"])], x="Month", y="Quantity_Allotted_NFSA_Qtls", hue="Commodity_Name", marker="o", linewidth=2)
plt.title("Staple Foodgrain Monthly Allotment (Rice vs Ragi)")
plt.ylabel("Total Quantity (Quintals)")
plt.show()

## 6. Correlation Analysis between Infrastructure and Utilization
We compute a correlation matrix to see how the number of active shops relates to card volume, member density, and lifting percentage.

In [ ]:
corr_cols = ["No_of_Active_FPS", "No_of_Ration_Cards", "No_of_Ration_Cards_Taken", "No_of_Members_Taken", "Lifting_Percentage", "Quantity_Allotted_NFSA_Qtls"]
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of PDS Statistics")
plt.show()

## 7. eKYC Pending Backlog by Taluk
We identify the average absolute number of active beneficiaries who have *not* completed their Aadhaar eKYC, helping administrators target verification drives.

In [ ]:
df["Pending_eKYC"] = df["No_of_Members_Taken"] - df["No_of_Members_Taken_eKYC"]
backlog = df.groupby("Taluk_Name")["Pending_eKYC"].mean().reset_index().sort_values(by="Pending_eKYC", ascending=False)

print("Average Pending eKYC Members per Monthly Report:")
print(backlog.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=backlog, x="Pending_eKYC", y="Taluk_Name", palette="Oranges")
plt.title("Average Pending eKYC Members by Taluk")
plt.xlabel("Pending eKYC Members")
plt.show()

## 8. Regional Staple Preference (Ragi-to-Rice Ratio)
Different taluks have varying dietary patterns. We calculate the ratio of Ragi allotment to Rice allotment to measure regional food grain preference.

In [ ]:
allot_sums = df.groupby(["Taluk_Name", "Commodity_Name"])["Quantity_Allotted_NFSA_Qtls"].sum().unstack()
allot_sums["Ragi_to_Rice_Ratio"] = allot_sums["Ragi"] / allot_sums["Rice"]
ragi_preference = allot_sums["Ragi_to_Rice_Ratio"].reset_index().sort_values(by="Ragi_to_Rice_Ratio", ascending=False)

print("Ragi to Rice Allotment Ratio:")
print(ragi_preference.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=ragi_preference, x="Ragi_to_Rice_Ratio", y="Taluk_Name", palette="YlOrBr")
plt.title("Staple Ratio (Ragi Allotment / Rice Allotment) by Taluk")
plt.xlabel("Ragi-to-Rice Ratio")
plt.show()

## 9. Monthly Seasonality of Ration Lifting Success
Is the transaction success rate higher during festive months (e.g. Sept/Oct/Dec) compared to summer months?

In [ ]:
monthly_lifting = df.groupby("Month")["Lifting_Percentage"].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly_lifting, x="Month", y="Lifting_Percentage", marker="s", color="crimson", linewidth=2.5)
plt.title("Monthly Trend in Average Lifting Success Rate (%)")
plt.ylabel("Average Lifting Percentage (%)")
plt.show()

## 10. Grain Load distributed per Active Ration Card
We calculate the average quantity of NFSA grain allotted per active cardholder:
$$\text{Grain per Card} = \frac{\text{Quantity Allotted NFSA}}{\text{No. of Ration Cards Taken}}$$

In [ ]:
active_rows = df[df["No_of_Ration_Cards_Taken"] > 0].copy()
active_rows["Grain_Per_Card"] = active_rows["Quantity_Allotted_NFSA_Qtls"] / active_rows["No_of_Ration_Cards_Taken"]
grain_load = active_rows.groupby("Taluk_Name")["Grain_Per_Card"].mean().reset_index().sort_values(by="Grain_Per_Card", ascending=False)

print("Average NFSA Grain Allotment (Quintals per Active Card):")
print(grain_load.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=grain_load, x="Grain_Per_Card", y="Taluk_Name", palette="teal")
plt.title("Average NFSA Grain Load per Active Ration Card (Quintals)")
plt.xlabel("Grain load (Quintals)")
plt.show()

## 11. Average Family Size (Members per Ration Card)
We calculate the average family size of families utilizing PDS benefits across taluks:
$$\text{Family Size} = \frac{\text{No. of Members Taken}}{\text{No. of Ration Cards Taken}}$$

In [ ]:
active_rows["Family_Size"] = active_rows["No_of_Members_Taken"] / active_rows["No_of_Ration_Cards_Taken"]
family_summary = active_rows.groupby("Taluk_Name")["Family_Size"].mean().reset_index().sort_values(by="Family_Size", ascending=False)

print("Average Family Size (Beneficiaries per Card):")
print(family_summary.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=family_summary, x="Family_Size", y="Taluk_Name", palette="rocket")
plt.title("Average Beneficiary Family Size by Taluk")
plt.xlabel("Members per Card")
plt.show()

## 12. Statistical Outliers in Monthly Allotment Volumes
We identify months and taluks with unusually high or low allotments using Z-score calculation on the NFSA allotment quantities.

In [ ]:
# Calculate Z-score for NFSA Allotment
df["Allotment_ZScore"] = stats.zscore(df["Quantity_Allotted_NFSA_Qtls"])
outliers = df[np.abs(df["Allotment_ZScore"]) > 2.0][["Month", "Year", "Taluk_Name", "Commodity_Name", "Quantity_Allotted_NFSA_Qtls", "Allotment_ZScore"]]

print(f"Found {len(outliers)} statistical outlier records (Z-score > 2):")
print(outliers.sort_values(by="Quantity_Allotted_NFSA_Qtls", ascending=False).to_string(index=False))

## 13. Stability of PDS Distribution (Variance in Lifting Rate)
A lower standard deviation of the lifting rate over the 12 months represents a stable, reliable distribution network, while high standard deviation indicates supply instability.

In [ ]:
stability = df.groupby("Taluk_Name")["Lifting_Percentage"].std().reset_index().rename(columns={"Lifting_Percentage": "Lifting_Std_Dev"})
stability = stability.sort_values(by="Lifting_Std_Dev")

print("Ration Distribution Stability (Lower Std Dev = Higher Stability):")
print(stability.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=stability, x="Lifting_Std_Dev", y="Taluk_Name", palette="mako")
plt.title("Ration Lifting Percentage Standard Deviation (Lower is More Stable)")
plt.xlabel("Standard Deviation (%)")
plt.show()

## 14. Monthly Trend of Active Fair Price Shops (FPS)
We monitor whether the number of active shops is fluctuating or remains constant over the 12-month period.

In [ ]:
monthly_shops = df.groupby("Month")["No_of_Active_FPS"].max().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly_shops, x="Month", y="No_of_Active_FPS", color="skyblue")
plt.title("Active Fair Price Shops (FPS) Trend by Month")
plt.ylabel("Number of Active Shops")
plt.ylim(1000, 1025) # Focus on the change
plt.show()

## 15. Total District-level Combined Foodgrain Footprint (Monthly Allotment sum)
We calculate the combined monthly foodgrain footprint (NFSA allotment sum for Rice + Ragi + Wheat + Sugar) distributed across the entire Mysuru district.

In [ ]:
district_footprint = df.groupby("Month")["Quantity_Allotted_NFSA_Qtls"].sum().reset_index()
print("District Monthly Combined Foodgrain footprint (Quintals):")
print(district_footprint.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=district_footprint, x="Month", y="Quantity_Allotted_NFSA_Qtls", palette="cubehelix")
plt.title("Mysuru District Combined Monthly PDS Footprint (Quintals)")
plt.ylabel("Combined Quantity (Quintals)")
plt.show()